# RAG Retrieval Baseline Comparison

این notebook سه روش retrieval را تست می‌کند:

1. BM25 فقط
2. Embedding فقط
3. Hybrid (بر اساس config)

تمام component ها از src import می‌شوند و notebook فقط برای تست و visualization است.


In [31]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.rag.config import load_config

from src.rag.retrieval.factory import RetrieverFactory
from src.rag.retrieval.hybrid import HybridRetriever

from src.rag.embedding.factory import EmbeddingFactory
from src.rag.utils.text import build_comment_text


## Load data

دیتای processed شده را اینجا وارد می‌کنیم.

مسیر را مطابق ساختار پروژه تنظیم کن.


In [33]:
# مثال:
comments = pd.read_parquet("../data/processed/comments_clean.parquet")
products = pd.read_parquet("../data/processed/products_clean.parquet")

# اگر در kernel قبلی load شده باشد این سلول فقط برای بررسی است

comments.head()


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate,created_at_gregorian
0,14144758,هری پاتر,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,10 آذر 1399,5.0,recommended,True,1075274,NaN,NaN,1136,8,بیگای استودیو,5AADE,NaN,2020-11-30
1,41782279,روغن ریش,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,21 آبان 1401,3.0,recommended,True,6081008,NaN,NaN,896,79,گراندو بیوتی,6XN5S,NaN,2022-11-12
2,49569443,تقلبی,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,3 خرداد 1402,1.0,not_recommended,True,10545754,NaN,NaN,625,60,کافه سرگرمی,AXVST,NaN,2023-05-24
3,43932524,یک نظر بی اغراق!,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,15 دی 1401,5.0,recommended,True,4153832,NaN,NaN,524,51,گروه آروند,5AD65,NaN,2023-01-05
4,21396693,NaN,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,9 خرداد 1400,3.0,no_idea,True,2185657,NaN,NaN,450,2,تاباتا,C93UM,NaN,2021-05-30


In [34]:
config = load_config("../configs/rag.yaml")

config


{'retrieval': {'type': 'hybrid',
  'top_k': 5,
  'hybrid': {'bm25_weight': 0.3, 'embedding_weight': 0.7}},
 'embedding': {'provider': 'sentence_transformer',
  'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}}

## Prepare retrieval documents

In [35]:
product_id = 5715766

product_comments = comments[
    comments["product_id"] == product_id
].copy()


product_comments["search_text"] = (
    product_comments.apply(
        build_comment_text,
        axis=1
    )
)


product_comments.head()


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate,created_at_gregorian,search_text
258208,54113259,NaN,خیلی ضدآفتاب خوبی هست، من قبلا کرمهای خارجی بر...,7 مهر 1402,5.0,recommended,True,5715766,NaN,NaN,2,1,گالری کهن,AATCE,NaN,2023-09-29,خیلی ضدآفتاب خوبی هست، من قبلا کرمهای خارجی بر...
278842,53974772,اصلا توصیه نمیکنم,حتی همسرم که پوستش جوش نمیزنه با این محصول جوش...,2 مهر 1402,1.0,not_recommended,True,5715766,NaN,NaN,3,0,دیجی‌کالا,5A52N,NaN,2023-09-24,اصلا توصیه نمیکنم حتی همسرم که پوستش جوش نمیزن...
384491,54075631,عالی,بدون بو اصلا چشم رو نمیسوزونه یکم رنگ پوستو رو...,5 مهر 1402,5.0,recommended,True,5715766,NaN,NaN,0,2,دیجی‌کالا,5A52N,NaN,2023-09-27,عالی بدون بو اصلا چشم رو نمیسوزونه یکم رنگ پوس...
422922,54171565,افتضاح,واقعا افتضاح و بیخوده من دلیل وجود اییییییین ه...,9 مهر 1402,1.0,not_recommended,True,5715766,NaN,NaN,0,2,گالری کهن,AATCE,NaN,2023-10-01,افتضاح واقعا افتضاح و بیخوده من دلیل وجود اییی...
423012,54109093,NaN,خیلی سبک و خوبه خیلی وقته استفاده میکنم راضیم ...,6 مهر 1402,5.0,recommended,True,5715766,NaN,NaN,1,1,دیجی‌کالا,5A52N,NaN,2023-09-28,خیلی سبک و خوبه خیلی وقته استفاده میکنم راضیم ...


In [36]:
query = "آیا برای پوست چرب مناسب است؟"
top_k = 5


## 1) BM25 only

In [37]:
bm25_retriever = RetrieverFactory.create(
    "bm25",
    product_comments
)


bm25_results = bm25_retriever.retrieve(
    query,
    top_k=top_k
)


bm25_results[
    ["body", "rate", "score"]
]


,body,rate,score
108,اگر پوست چرب دارید خیلی مناسب نیست چون برق میو...,2.0,8.239756
153,ضدآفتاب خیلی خوبیه برای پوست چرب,4.0,7.206134
220,برای پوست چرب عالیه,5.0,7.206134
59,قیمت مناسب در دیجی کالا,5.0,7.042508
125,برای پوست چرب نمیتونم بگم عالیه ولی خوبه,4.0,6.666359


## 2) Embedding only

In [39]:
embedding_model = EmbeddingFactory.create(
    provider=config["embedding"]["provider"],
    model_name=config["embedding"]["model"]
)


embedding_retriever = RetrieverFactory.create(
    "embedding",
    product_comments,
    embedding_model=embedding_model
)


embedding_results = embedding_retriever.retrieve(
    query,
    top_k=top_k
)


embedding_results[
    ["body", "rate", "score"]
]


Loading weights: 100%|████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 17431.74it/s]


,body,rate,score
220,برای پوست چرب عالیه,5.0,0.840207
153,ضدآفتاب خیلی خوبیه برای پوست چرب,4.0,0.831016
43,برای پوست چرب خوب بود,4.0,0.792578
196,سبک و خوبه ولی برعکس نظر دوستان صورت منو چرب م...,4.0,0.779009
116,به خاطر بافت فلوییدی و سبکش واقعا برای پوست ها...,5.0,0.759629


## 3) Hybrid from config

In [40]:
bm25_retriever = RetrieverFactory.create(
    "bm25",
    product_comments
)


embedding_retriever = RetrieverFactory.create(
    "embedding",
    product_comments,
    embedding_model=embedding_model
)


hybrid_cfg = config["retrieval"]["hybrid"]


hybrid_retriever = HybridRetriever(
    bm25_retriever,
    embedding_retriever,
    bm25_weight=hybrid_cfg["bm25_weight"],
    embedding_weight=hybrid_cfg["embedding_weight"]
)


hybrid_results = hybrid_retriever.retrieve(
    query,
    top_k=top_k
)


hybrid_results


,body,bm25_score,embedding_score,score
5,برای پوست چرب عالیه,0.748859,1.000000,0.924658
15,ضدآفتاب خیلی خوبیه برای پوست چرب,0.748859,0.951122,0.890443
4,برای پوست چرب خوب بود,0.406671,0.746697,0.644689
14,سبک و خوبه ولی برعکس نظر دوستان صورت منو چرب م...,0.277953,0.674537,0.555562
0,اگر پوست چرب دارید خیلی مناسب نیست چون برق میو...,1.000000,0.265045,0.485532


## Comparison

سه خروجی بالا را برای ارزیابی کیفیت retrieval مقایسه می‌کنیم.
